In [13]:
from khmernormalizer import normalize
from khmercut import tokenize
from khmerpunctuate import punctuate

text = "អយ្យការអមសាលាដំបូងរាជធានីភ្នំពេញបានព្រមានថានឹងចេញដីកាបញ្ជាឲ្យបង្ខំនិងឲ្យឃុំខ្លួនតាមនីតិវិធីប្រសិនបើលោករ៉ុងឈុនដែលបច្ចុប្បន្នជាទីប្រឹក្សាគណបក្សកម្លាំងជាតិមិនបានបង់ប្រាក់ពិន័យចំនួន២លានរៀលឲ្យបានមុនថ្ងៃទី០៤ខែមីនាឆ្នាំ២០២៤ទេនោះ"
import re

def space_english_words(sentence):
    # Add a space before any English word that's adjacent to Khmer characters
    spaced = re.sub(r'(?<=[\u1780-\u17DD\u17E0-\u17FF])(?=[A-Za-z])', ' ', sentence)
    # Add a space after any English word that's adjacent to Khmer characters
    spaced = re.sub(r'(?<=[A-Za-z])(?=[\u1780-\u17DD\u17E0-\u17FF])', ' ', spaced)
    spaced = re.sub(r"(\w)([A-Z])", r"\1 \2",spaced)
    return spaced

def remove_trailing_periods(text):
    # Use regular expression to remove trailing periods
    cleaned_text = re.sub(r'\.+$', '', text)
    return cleaned_text

def add_double_space_after_period(text):
    # Replace any number of spaces after a full stop with two spaces
    # The regex looks for a period followed by any number of whitespace characters
    # and replaces it with a period followed by exactly two spaces
    new_text = re.sub(r'\។\s*', '។ ', text)
    
    # Ensure no double space at the end of the text if it ends with a period
    if new_text.endswith(".  "):
        new_text = new_text[:-1]
    
    return new_text

def remove_consecutive_khmer_periods(text):
    # Replace two consecutive Khmer periods with a single one
    return re.sub(r'។{2,}', '។', text)

def space_restoration(text):
    text = normalize(text)
    tokens = tokenize(text)

    if not tokens:
        return text 

    output_text = ""
    for token, punct, punct_id in punctuate(tokens):
        # exclude special tokens like I-NUMBER, B-NUMBER, I-QUOTE and B-QUOTE
        if punct_id < 7:
            output_text += token + punct
        else:
            output_text += token

    return space_english_words(output_text)

answer = space_restoration(text)
print(answer)


អយ្យការអមសាលាដំបូងរាជធានីភ្នំពេញ បានព្រមានថា នឹងចេញដីកាបញ្ជាឱ្យបង្ខំ និងឱ្យឃុំខ្លួនតាមនីតិវិធី ប្រសិនបើលោក រ៉ុង ឈុន ដែលបច្ចុប្បន្នជាទីប្រឹក្សាគណបក្សកម្លាំងជាតិ មិនបានបង់ប្រាក់ពិន័យចំនួន២លានរៀលឱ្យបានមុនថ្ងៃទី០៤ខែមីនា ឆ្នាំ២០២៤ទេនោះ


In [14]:
import pandas as pd
from khmerpunctuate import punctuate
from khmernormalizer import normalize
from khmercut import tokenize
import re

df = pd.read_csv("Final_test_dataset.csv")
df = df.dropna()

def clean_irrelevant(result):    
    result = result.replace('u200b','')
    result = result.replace('u 200 b','')
    result = result.replace('u200c','')
    result = result.replace('u 200 c','')
    result = result.replace('ufeff ','')
    result = result.replace('ufeff','')
    # result = result.replace('quot ','')

    return result

def remove_space(text):
    text_without_spaces = re.sub(r'\s+', '', text)
    text = clean_irrelevant(text_without_spaces)
    return text

df["Khmer"] = df["Khmer"].apply(remove_space)
df


,Khmer,English
0,បើទោះជាមានក្រុមហ៊ុនច្រើនក្នុងរាជធានីភ្នំពេញដែល...,Although there are more companies in Phnom Pen...
1,លោកអូមរិទ្ធីរិទ្ធមានប្រសាសន៍ថាលោកនៅតែមានសុទិដ្...,Omrithy Rithy said he was still optimistic abo...
2,ហេតុដូចម្តេចទើបមានការបង្កើតតាក់ស៊ីក្រហមឬTaxiRo...,Why has it to be creating taxi or Taxi Rouge up?
3,និយាយទៅខ្ញុំបានសង្កេតពីការវិវត្តរបស់ប្រទេសកម្ព...,Talk to I observed the development of Cambodia...
4,ខ្ញុំតែងមានគំនិតថានឹងអាចធ្វើអ្វីមួយដែលអាចបង្កើ...,I always have idea that it can do something th...
...,...,...
10356,២០ដូច្នេះចូរ​ឲ្យ​បណ្ដាំ​របស់​ព្រះ​ដឹក​នាំ​ប្អូ...,"20 Let the word of God guide you, and do the t..."
10357,សេវាSathapana,Sathapana Services
10358,សីលាបើមិនធ្វើមានតែអត់។,"Seila, if not, there is only no."
10359,សំនួរការទាញយកhighdefinitionenvironmentv21.zip,Download question highdefinitionenvironmentv21...


In [15]:
column_name = 'Khmer'

# Specify the output text file path
output_file_path = 'Khmer2.txt'

# Write the column to the text file, one line at a time
with open(output_file_path, 'w', encoding='utf-8') as file:
    for line in df[column_name]:
        if len(line)<150:
            file.write(f"{(line)}\n")

print(f"Column '{column_name}' has been written to {output_file_path}")

Column 'Khmer' has been written to Khmer2.txt
